# C.1 寫出中間步驟能幫忙嗎？


對「2+3=?」，直接答5與寫「從2開始數三次：3、4、5，所以是5」都可能正確。後者把一種解題方法展開，讀者可以看見它怎麼到5；但比較長也會增加生成成本，且步驟本身可能錯。前置是[逐token生成](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html)與[對話SFT](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)：模型能被示範資料教成產生不同回答形式，形式與正確率要分開評估。

把問題固定，訓練目標可以是短答案，也可以是含步驟的答案。在SFT中，兩者都是要逐項預測的回答token，只是文字內容和長度不同。為什麼步驟可能有幫助？模型後續生成可以讀取前面已寫出的中間結果，形成可檢查的分段解法；同時，中間結果若錯了，也可能把後面帶錯。這是可能的作用機制，不能光看一份人寫的示例就宣稱模型已獲益。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import ByteTokenizer

tokenizer = ByteTokenizer()
examples = [
    {"question": "2+3=?", "answer": "5"},
    {"question": "2+3=?", "answer": "從2開始數三次：3、4、5，所以是5。"},
]
for example in examples:
    ids = tokenizer.encode(example["answer"])
    print(example["question"], example["answer"], "回答token數", len(ids))

輸入是兩筆相同問題、不同回答的人工資料。ByteTokenizer把UTF-8的每個byte變成一個ID，`encode`得到ID清單，`len`數清單長度；這不是用中文字數假裝token數。第一份回答有1個token，第二份有47個。本例未加入對話標記或EOS，所以是回答文字本身的長度；其他tokenizer的切分不同，不能套用47這個數字。

程式只是檢查資料設計，沒有訓練兩個模型或測出推理能力。真正比較時，可以從相同起點出發，分別用短答與步驟資料訓練，保留相同題目分布與最後真值，再評估沒出現在訓練裡的新數字題。除了答案正確率，也要記錄生成token數和時間；若步驟版本多用許多訓練token，還需說明比較是在相同題數還是相同計算預算下做的。

可見的步驟文字也不等於模型內部運算的完整紀錄。文字可能是合理解釋、錯誤推導，甚至與最後答案沒有可靠關係。下一節會用「最後答對、第一步仍錯」的例子實際核對，避免把流暢長文當成正確保證。

練習只把第二份answer中的最後5改成6，保留前面數到5的步驟。先預測ByteTokenizer長度仍47，但結論與前文不一致，再執行核對。你應能指出程式只報長度，沒有判正誤；若要比較推理品質，必須另加答案與步驟的檢查。
